# Barcode Detection with Quagga2

This notebook tests `@ericblade/quagga2` as an atomic JavaScript decoder against the 300 DPI rendered page cache. It keeps Quagga as the decoder, but feeds it full pages plus simple high-aspect crop candidates so tiny 1D barcodes get a fair pass.


In [ ]:
from pathlib import Path
import json
import math
import shutil
import subprocess

import cv2
from PIL import Image, ImageDraw

try:
    from IPython.display import display
except ImportError:
    def display(image):
        return image


def find_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / 'data' / 'pdfs').exists():
            return path
    raise RuntimeError('Could not find repo root containing data/pdfs')


ROOT = find_root()
PDF_PATH = ROOT / 'data' / 'pdfs' / 'Quality Dossier.pdf'
PAGES = list(range(1, 22))
OUT_DIR = ROOT / 'output' / 'quagga-atom'
PAGE_DIR = OUT_DIR / 'rendered-pages'
DPI = 300
CROP_DIR = OUT_DIR / 'crops'
NPM_PREFIX = OUT_DIR
NODE_MODULE = NPM_PREFIX / 'node_modules' / '@ericblade' / 'quagga2'
RUNNER = OUT_DIR / 'quagga-decode.js'
GRID_COLS = 2
CELL_WIDTH = 600
READERS = [
    'code_39_reader', 'code_128_reader', 'code_93_reader',
    'i2of5_reader', 'codabar_reader', 'ean_reader',
    'ean_8_reader', 'upc_reader', 'upc_e_reader',
]

def find_pdftoppm():
    found = shutil.which('pdftoppm')
    if found:
        return found
    for candidate in [Path('/opt/homebrew/bin/pdftoppm'), Path('/usr/local/bin/pdftoppm')]:
        if candidate.exists():
            return str(candidate)
    cache_root = Path.home() / '.cache' / 'codex-runtimes'
    if cache_root.exists():
        for candidate in cache_root.glob('**/pdftoppm'):
            if candidate.is_file():
                return str(candidate)
    raise RuntimeError('pdftoppm not found. Install Poppler or add pdftoppm to PATH.')


def render_pdf_if_needed():
    PAGE_DIR.mkdir(parents=True, exist_ok=True)
    marker = PAGE_DIR / '.dpi'
    pages_ready = all((PAGE_DIR / f'page-{page:02d}.png').exists() for page in PAGES)
    dpi_ready = marker.exists() and marker.read_text().strip() == str(DPI)
    if pages_ready and dpi_ready:
        return
    for old_page in PAGE_DIR.glob('page-*.png'):
        old_page.unlink()
    subprocess.run(
        [find_pdftoppm(), '-r', str(DPI), '-png', str(PDF_PATH), str(PAGE_DIR / 'page')],
        check=True,
    )
    marker.write_text(str(DPI), encoding='utf-8')



Install the maintained Quagga package into the output folder and write a tiny Node runner for `decodeSingle`.


In [ ]:
def ensure_quagga():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    CROP_DIR.mkdir(exist_ok=True)
    if shutil.which('node') is None or shutil.which('npm') is None:
        raise RuntimeError('Quagga needs node and npm on PATH.')
    if not NODE_MODULE.exists():
        subprocess.run(
            ['npm', '--prefix', str(NPM_PREFIX), 'install', '@ericblade/quagga2@1.12.1'],
            check=True,
        )
    RUNNER.write_text("""const fs = require('fs');
const Quagga = require('./node_modules/@ericblade/quagga2').default;

const args = JSON.parse(fs.readFileSync(0, 'utf8'));

function decode(src, reader, locate) {
  return new Promise((resolve) => {
    Quagga.decodeSingle({
      src,
      numOfWorkers: 0,
      inputStream: { size: 0 },
      locator: { patchSize: 'medium', halfSample: false },
      locate,
      decoder: { readers: [reader] },
    }, (result) => resolve(result && result.codeResult ? result : null));
  });
}

(async () => {
  for (const reader of args.readers) {
    const result = await decode(args.image, reader, args.locate);
    if (result) {
      process.stdout.write(JSON.stringify({
        reader,
        codeResult: result.codeResult,
        box: result.box || null,
        line: result.line || null,
      }));
      return;
    }
  }
  process.stdout.write('null');
})().catch((err) => {
  console.error(err && err.stack ? err.stack : err);
  process.exit(1);
});
""", encoding='utf-8')


def run_quagga(image_path, locate=True):
    ensure_quagga()
    payload = json.dumps({'image': str(image_path), 'readers': READERS, 'locate': locate})
    proc = subprocess.run(
        ['node', str(RUNNER)],
        input=payload,
        text=True,
        capture_output=True,
        check=True,
    )
    return json.loads(proc.stdout or 'null')


Build small barcode-like crop candidates and normalize Quagga results into the same detection shape as the other atom notebooks.


In [ ]:
def candidate_boxes(image_path, limit=24):
    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_RECT, (35, 7)))
    grad = cv2.Sobel(blackhat, cv2.CV_32F, 1, 0, ksize=-1)
    grad = cv2.convertScaleAbs(grad)
    grad = cv2.GaussianBlur(grad, (9, 9), 0)
    _, thresh = cv2.threshold(grad, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (35, 9)))
    closed = cv2.erode(closed, None, iterations=1)
    closed = cv2.dilate(closed, None, iterations=2)

    h_img, w_img = gray.shape
    boxes = []
    for contour in cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]:
        x, y, w, h = cv2.boundingRect(contour)
        if w < 120 or h < 20 or w / max(h, 1) < 3:
            continue
        pad_x, pad_y = int(w * 0.12) + 20, int(h * 0.8) + 20
        x1, y1 = max(0, x - pad_x), max(0, y - pad_y)
        x2, y2 = min(w_img, x + w + pad_x), min(h_img, y + h + pad_y)
        boxes.append((x1, y1, x2, y2))

    boxes.sort(key=lambda b: (b[1], b[0], -(b[2] - b[0]) * (b[3] - b[1])))
    return boxes[:limit]


def result_box(result, crop_box):
    x1, y1, x2, y2 = crop_box
    points = result.get('box') or []
    if len(points) >= 4:
        return [[float((p.get('x', p.get('0')) if isinstance(p, dict) else p[0]) + x1), float((p.get('y', p.get('1')) if isinstance(p, dict) else p[1]) + y1)] for p in points[:4]]
    return [[float(x1), float(y1)], [float(x2), float(y1)], [float(x2), float(y2)], [float(x1), float(y2)]]


def add_unique(detections, det):
    key = (det['format'], det['raw_value'])
    if key not in {(d['format'], d['raw_value']) for d in detections}:
        detections.append(det)


def read_barcodes(image_path, page):
    CROP_DIR.mkdir(parents=True, exist_ok=True)
    image = Image.open(image_path).convert('RGB')
    full_box = (0, 0, image.width, image.height)
    detections = []

    for i, box in enumerate([full_box, *candidate_boxes(image_path)]):
        crop_path = CROP_DIR / f'page-{page:02d}-crop-{i:02d}.png'
        image.crop(box).save(crop_path)
        result = run_quagga(crop_path, locate=(i == 0))
        if not result:
            continue
        code = result['codeResult']
        add_unique(detections, {
            'format': code.get('format') or result.get('reader'),
            'raw_value': code.get('code') or '',
            'confidence': float(code.get('confidence') or 0),
            'box': result_box(result, box),
            'source': 'full-page' if i == 0 else 'candidate-crop',
        })
    return detections


Draw detections and assemble the review grid.


In [ ]:
def draw_detections(image_path, detections):
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    colors = ['red', 'lime', 'blue', 'orange', 'magenta', 'cyan', 'yellow']
    for i, det in enumerate(detections, start=1):
        points = [tuple(point) for point in det['box']]
        color = colors[(i - 1) % len(colors)]
        draw.line(points + [points[0]], fill=color, width=6)
        x = min(p[0] for p in points)
        y = max(0, min(p[1] for p in points) - 26)
        label = f'{i} {det["format"]}: {det["raw_value"][:28]}'
        draw.rectangle([x, y, x + min(560, max(220, 8 * len(label))), y + 26], fill=color)
        draw.text((x + 4, y + 5), label, fill='black')
    return image


def create_grid(images, cols=2, cell_width=600):
    thumbs = []
    for image in images:
        ratio = cell_width / image.width
        thumbs.append(image.resize((cell_width, int(image.height * ratio))))
    rows = math.ceil(len(thumbs) / cols)
    cell_height = max(image.height for image in thumbs)
    grid = Image.new('RGB', (cols * cell_width, rows * cell_height), 'white')
    for i, image in enumerate(thumbs):
        grid.paste(image, ((i % cols) * cell_width, (i // cols) * cell_height))
    return grid


Run Quagga over all pages and save JSON plus preview images.


In [ ]:
render_pdf_if_needed()
ensure_quagga()
OUT_DIR.mkdir(parents=True, exist_ok=True)

results = []
preview_images = []
total_found = 0

for page in PAGES:
    image_path = PAGE_DIR / f'page-{page:02d}.png'
    if not image_path.exists():
        print(f'Skipping page {page}: image not found')
        continue

    detections = read_barcodes(image_path, page)
    total_found += len(detections)
    preview = draw_detections(image_path, detections)
    preview_path = OUT_DIR / f'page-{page:02d}.png'
    preview.save(preview_path)
    preview_images.append(preview)

    print(f'page {page:02d}: found {len(detections)}')
    results.append({
        'page': page,
        'count': len(detections),
        'detections': detections,
        'debug_image': str(preview_path),
    })

(OUT_DIR / 'detections.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
print(f'\nTotal: {total_found} barcodes across {len(results)} pages')
print(f'Saved JSON: {OUT_DIR / "detections.json"}')

if preview_images:
    grid = create_grid(preview_images, cols=GRID_COLS, cell_width=CELL_WIDTH)
    grid_path = OUT_DIR / 'all-pages-grid.png'
    grid.save(grid_path)
    print(f'Saved grid: {grid_path}')
    display(grid)
else:
    print('No pages processed.')

